# 검증 프로토콜 V2 결과 보고서

**TL;DR:** 같은 KFold를 반복해서 본 선택 과적합을 줄이기 위해 중복 입력 행을 묶은 Stratified Group 5-Fold와 신규 Audit Seed를 사용했습니다. V3가 평균 기준으로 가장 좋지만 Audit paired-bootstrap 95% CI가 0을 포함하므로 아직 확실한 승격 후보는 아닙니다. Test 데이터는 사용하지 않았습니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
if not (RESULTS / 'audit_summary.csv').exists():
    RESULTS = ROOT / 'validation' / 'results'
audit = pd.read_csv(RESULTS / 'audit_summary.csv')
development = pd.read_csv(RESULTS / 'development_summary.csv')
with (RESULTS / 'data_profile.json').open(encoding='utf-8') as f:
    profile = json.load(f)
print(profile)

## 비교 결과

후보별 평균 MAE뿐 아니라 V1 대비 쌍대 오차 차이와 그룹 Bootstrap 신뢰구간을 함께 확인합니다.

In [ ]:
columns = [
    'stage', 'model', 'mean_mae', 'mean_delta_vs_v1',
    'seed_win_rate_vs_v1', 'paired_delta_ci95_lower',
    'paired_delta_ci95_upper',
]
comparison = pd.concat([development[columns], audit[columns]], ignore_index=True)
comparison.sort_values(['stage', 'mean_mae'])

In [ ]:
v3_audit = audit.loc[audit['model'].eq('v3')].iloc[0]
assert v3_audit['seed_win_rate_vs_v1'] == 1.0
assert v3_audit['mean_delta_vs_v1'] < 0
assert v3_audit['paired_delta_ci95_upper'] > 0  # 아직 강한 승격 기준 미통과
print('V3: 유망하지만 Audit 95% CI가 0을 포함하여 승격 보류')

## 판단

- V3: Development/Audit 평균 1위, Audit Seed 3/3 승리. 다만 Audit CI 상한이 +0.000079라서 확실한 개선으로 단정하지 않습니다.
- V2: CV에서는 좋아졌지만 실제 Public은 악화되었습니다. 작은 CV 차이가 리더보드 향상을 보장하지 않는 사례입니다.
- 다음 후보: ExtraTrees를 90~99% 유지하고, 오차 상관이 낮은 보조 예측을 1~10%만 혼합하되 비율은 Train-only OOF에서 고정합니다.